In [ ]:
import os, shutil, rasterio, sys
sys.path.append('backend/app/')
from rasterio.features import shapes
from shapely.geometry import shape
import geopandas as gpd, pandas as pd
import numpy as np
from backend.app.services import flow_functions

In [ ]:
catchment_path = r'backend\src\flow_samples\catchment.geojson'
terrain_path = r'backend\src\flow_samples\dtm10.tif'
soil_path = r'backend\src\flow_samples\soil.geojson'
land_path = r'backend\src\flow_samples\landcover.geojson'
river_path = r'backend\src\flow_samples\river.geojson'
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
catchment = gpd.read_file(catchment_path)
terrain = rasterio.open(terrain_path)
soil = gpd.read_file(soil_path)
land = gpd.read_file(land_path)
river = gpd.read_file(river_path)
catchment

c:\Envs\hyd_ai\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'width' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)
c:\Envs\hyd_ai\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'depth' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)
c:\Envs\hyd_ai\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'manning_n' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


,id,_id,width,depth,manning_n,river,geometry
0,0,1,None,None,None,NaN,"LINESTRING (6.40751 62.4574, 6.40715 62.45729,..."
1,1,2,None,None,None,NaN,"LINESTRING (6.40751 62.4574, 6.40759 62.45714,..."
2,2,3,None,None,None,NaN,"LINESTRING (6.40786 62.45824, 6.40791 62.45806..."
3,3,4,None,None,None,NaN,"LINESTRING (6.40786 62.45824, 6.40824 62.45826..."
4,4,5,None,None,None,NaN,"LINESTRING (6.47504 62.46411, 6.47504 62.46411)"
5,5,6,None,None,None,NaN,"LINESTRING Z (6.42709 62.46721 0, 6.42712 62.4..."
6,6,7,None,None,None,NaN,"LINESTRING Z (6.45569 62.4704 0, 6.45588 62.47..."
7,7,8,None,None,None,NaN,"LINESTRING (6.38483 62.4616, 6.38467 62.4615, ..."
8,8,9,None,None,None,NaN,"LINESTRING (6.39584 62.46201, 6.3957 62.46182,..."
9,9,10,None,None,None,NaN,"LINESTRING (6.38483 62.4616, 6.38505 62.46152,..."


In [29]:
key, soil_path = "land", r"backend\src\flow_samples\landcover.tif"
if key == "soil": 
    folder, func_codes = "soils", flow_functions.soil_codes
    func_types = flow_functions.soil_types
    new_cols = ["theta_s", "theta_r", "k_sat_ver", "soil_depth", "conductivity_decay", "brooks_corey"]
elif key == "land": 
    folder, func_codes = "lands", flow_functions.land_codes
    func_types = flow_functions.land_types
    new_cols = ["LAI", "root_depth", "interception", "manning_n", "albedo", "kc"]

In [52]:
with rasterio.open(soil_path) as src:
    data = src.read(1)
    mask = data != src.nodata
    data = data.astype(np.int32)
    results = ({ "geometry": shape(geom), key: func_codes.get(value, "")
    } for geom, value in shapes(data, mask=mask, transform=src.transform))
    geoms = list(results)
del data
gdf = gpd.GeoDataFrame(geoms, crs=src.crs)

In [55]:
path = r"backend\src\flow_samples\landcover.geojson"
gdf = gpd.read_file(path)

In [56]:
if '_id' not in gdf.columns: gdf.insert(0, '_id', range(1, len(gdf) + 1))
if key not in gdf.columns: gdf.insert(1, key, 'None')
mapped = gdf[key].map(lambda x: func_types.get(x, ["None"] * len(new_cols)))
gdf[new_cols] = pd.DataFrame(mapped.tolist(), columns=new_cols)
gdf[key] = np.where(gdf[key]=='', 'None', gdf[key])
gdf[key] = gdf[key].astype(str)
gdf = gdf[['_id', key, 'geometry'] + new_cols]

In [57]:
gdf

,_id,land,geometry,LAI,root_depth,interception,manning_n,albedo,kc
0,1,None,"POLYGON ((6.37162 62.45823, 6.37162 62.45815, ...",None,None,None,None,None,None
1,2,Impervious/Urban,"POLYGON ((6.37229 62.45823, 6.37229 62.45815, ...",0.5,0.1,0.5,0.05,0.15,0.3
2,3,Shallow vegetation,"POLYGON ((6.37215 62.45831, 6.37215 62.45815, ...",2.0,0.5,1.0,0.15,0.23,0.9
3,4,None,"POLYGON ((6.37202 62.45831, 6.37202 62.45823, ...",None,None,None,None,None,None
4,5,Shallow vegetation,"POLYGON ((6.37162 62.45831, 6.37162 62.45823, ...",2.0,0.5,1.0,0.15,0.23,0.9
...,...,...,...,...,...,...,...,...,...
8832,8833,None,"POLYGON ((6.46464 62.48277, 6.46464 62.48269, ...",None,None,None,None,None,None
8833,8834,Shallow vegetation,"POLYGON ((6.46225 62.48301, 6.46225 62.48293, ...",2.0,0.5,1.0,0.15,0.23,0.9
8834,8835,Shallow vegetation,"POLYGON ((6.46424 62.4835, 6.46424 62.48342, 6...",2.0,0.5,1.0,0.15,0.23,0.9
8835,8836,Shallow vegetation,"POLYGON ((6.47582 62.48494, 6.47582 62.48486, ...",2.0,0.5,1.0,0.15,0.23,0.9


In [48]:
gdf

,_id,geometry,land,LAI,root_depth,interception,manning_n,albedo,kc
0,1,"POLYGON ((6.64177 62.52099, 6.64177 62.52091, ...",Dense vegetation,5.0,1.5,3.0,0.4,0.13,1.1
1,2,"POLYGON ((6.63032 62.52035, 6.63032 62.52027, ...",None,None,None,None,None,None,None
2,3,"POLYGON ((6.63045 62.52035, 6.63045 62.52027, ...",Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9
3,4,"POLYGON ((6.63099 62.52035, 6.63099 62.52027, ...",Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9
4,5,"POLYGON ((6.63112 62.52035, 6.63112 62.52027, ...",Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3
...,...,...,...,...,...,...,...,...,...
56649,56650,"POLYGON ((6.33183 62.41953, 6.33183 62.41945, ...",None,None,None,None,None,None,None
56650,56651,"POLYGON ((6.32917 62.41937, 6.32917 62.41929, ...",Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3
56651,56652,"POLYGON ((6.3297 62.41969, 6.3297 62.41961, 6....",Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9
56652,56653,"POLYGON ((6.3293 62.41937, 6.3293 62.41929, 6....",None,None,None,None,None,None,None


In [36]:
geoms

[{'geometry': <POLYGON ((6.642 62.521, 6.642 62.521, 6.642 62.521, 6.642 62.521, 6.642 62....>,
  'land': 'Dense vegetation'},
 {'geometry': <POLYGON ((6.63 62.52, 6.63 62.52, 6.63 62.52, 6.63 62.52, 6.63 62.52))>,
  'land': ''},
 {'geometry': <POLYGON ((6.63 62.52, 6.63 62.52, 6.631 62.52, 6.631 62.52, 6.63 62.52))>,
  'land': 'Shallow vegetation'},
 {'geometry': <POLYGON ((6.631 62.52, 6.631 62.52, 6.631 62.52, 6.631 62.52, 6.631 62.52))>,
  'land': 'Shallow vegetation'},
 {'geometry': <POLYGON ((6.631 62.52, 6.631 62.52, 6.631 62.52, 6.631 62.52, 6.631 62.52))>,
  'land': 'Impervious/Urban'},
 {'geometry': <POLYGON ((6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52))>,
  'land': 'Shallow vegetation'},
 {'geometry': <POLYGON ((6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52))>,
  'land': ''},
 {'geometry': <POLYGON ((6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52, 6.633 62.52))>,
  'land': 'Shallow vegetation'},
 {'geometry': <POLYGON ((6.64 62.521,

In [15]:
path = r"backend\src\flow_samples\landcover.geojson"
gdf = gpd.read_file(path)

In [16]:
gdf

,id,_id,land,LAI,root_depth,interception,manning_n,albedo,kc,geometry
0,0,1,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37162 62.45823, 6.37162 62.45815, ..."
1,1,2,Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3,"POLYGON ((6.37229 62.45823, 6.37229 62.45815, ..."
2,2,3,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37215 62.45831, 6.37215 62.45815, ..."
3,3,4,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37202 62.45831, 6.37202 62.45823, ..."
4,4,5,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37162 62.45831, 6.37162 62.45823, ..."
...,...,...,...,...,...,...,...,...,...,...
8832,8832,8833,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.46464 62.48277, 6.46464 62.48269, ..."
8833,8833,8834,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46225 62.48301, 6.46225 62.48293, ..."
8834,8834,8835,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46424 62.4835, 6.46424 62.48342, 6..."
8835,8835,8836,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.47582 62.48494, 6.47582 62.48486, ..."


In [14]:
gdf['land'] = gdf['land'].astype(str)
gdf

,id,_id,land,LAI,root_depth,interception,manning_n,albedo,kc,geometry
0,0,1,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37162 62.45823, 6.37162 62.45815, ..."
1,1,2,Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3,"POLYGON ((6.37229 62.45823, 6.37229 62.45815, ..."
2,2,3,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37215 62.45831, 6.37215 62.45815, ..."
3,3,4,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37202 62.45831, 6.37202 62.45823, ..."
4,4,5,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37162 62.45831, 6.37162 62.45823, ..."
...,...,...,...,...,...,...,...,...,...,...
8832,8832,8833,,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.46464 62.48277, 6.46464 62.48269, ..."
8833,8833,8834,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46225 62.48301, 6.46225 62.48293, ..."
8834,8834,8835,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46424 62.4835, 6.46424 62.48342, 6..."
8835,8835,8836,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.47582 62.48494, 6.47582 62.48486, ..."


In [ ]:
gdf['land'] = np.where(gdf['land']=='', 'None', gdf['land'])

In [11]:
gdf


,id,_id,land,LAI,root_depth,interception,manning_n,albedo,kc,geometry
0,0,1,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37162 62.45823, 6.37162 62.45815, ..."
1,1,2,Impervious/Urban,0.5,0.1,0.5,0.05,0.15,0.3,"POLYGON ((6.37229 62.45823, 6.37229 62.45815, ..."
2,2,3,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37215 62.45831, 6.37215 62.45815, ..."
3,3,4,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.37202 62.45831, 6.37202 62.45823, ..."
4,4,5,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.37162 62.45831, 6.37162 62.45823, ..."
...,...,...,...,...,...,...,...,...,...,...
8832,8832,8833,None,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((6.46464 62.48277, 6.46464 62.48269, ..."
8833,8833,8834,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46225 62.48301, 6.46225 62.48293, ..."
8834,8834,8835,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.46424 62.4835, 6.46424 62.48342, 6..."
8835,8835,8836,Shallow vegetation,2.0,0.5,1.0,0.15,0.23,0.9,"POLYGON ((6.47582 62.48494, 6.47582 62.48486, ..."
